# Gradient Field CNN Baseline for DeepFake Detection (v2)

Build a compact CNN that ingests gradient field representations and outputs:
1. A binary logit (real vs AI)
2. A 128-D embedding for future fusion

### Key Features (v2)
- **5-channel gradient field**: Gx, Gy, magnitude, angle, coherence
- **No per-sample normalization**: Preserves absolute magnitude differences
- **Coherence feature**: Measures local gradient alignment (discriminative!)
- **GPU-optimized**: All computation on GPU

### Mathematical Foundation
- **Luminance**: BT.709 (L = 0.2126R + 0.7152G + 0.0722B)
- **Gradients**: Sobel operators Gx, Gy
- **Magnitude**: |G| = √(Gx² + Gy²)
- **Angle**: θ = atan2(Gy, Gx)
- **Coherence**: κ = ((λ₁ - λ₂) / (λ₁ + λ₂))² from structure tensor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_DATA_DIR = "/content/drive/MyDrive/datasets"

In [ ]:
import os
import time
import math
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, roc_curve, precision_recall_curve,
    average_precision_score, classification_report
)
import numpy as np
from PIL import Image

if not torch.cuda.is_available():
    raise RuntimeError(
        '❌ CUDA not available! Training on CPU takes ~50 min/epoch.\n'
        'Go to Runtime → Change runtime type → Select GPU (T4 recommended)'
    )
print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


# Where to save best checkpoints & logs
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
data_dir = f"{GDRIVE_DATA_DIR}/OpenFake"
epochs = 30
batch_size = 64
lr = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"

# DataLoader config optimized for Colab
NUM_WORKERS = 2  # Colab max recommended
PIN_MEMORY = True  # Fast CPU→GPU transfer


# Early Stopping parameters
PATIENCE = 5
patience_counter = 0



print(f"Using device: {device}")

---
## 1. Metrics Dataclass

Clean, structured container for all evaluation metrics.

In [ ]:
@dataclass
class Metrics:
    """Container for all classification metrics."""
    accuracy: float = 0.0
    precision: float = 0.0
    recall: float = 0.0
    f1: float = 0.0
    specificity: float = 0.0
    auroc: float = 0.0
    avg_precision: float = 0.0  # Area under PR curve
    
    # Raw data for curve plotting
    labels: np.ndarray = field(default_factory=lambda: np.array([]))
    preds: np.ndarray = field(default_factory=lambda: np.array([]))
    probs: np.ndarray = field(default_factory=lambda: np.array([]))
    
    def __repr__(self):
        return (
            f"Metrics(acc={self.accuracy:.4f}, prec={self.precision:.4f}, "
            f"rec={self.recall:.4f}, f1={self.f1:.4f}, spec={self.specificity:.4f}, "
            f"auroc={self.auroc:.4f}, ap={self.avg_precision:.4f})"
        )
    
    def to_dict(self):
        """Return scalar metrics as dict (excludes arrays)."""
        return {
            'accuracy': self.accuracy,
            'precision': self.precision,
            'recall': self.recall,
            'f1': self.f1,
            'specificity': self.specificity,
            'auroc': self.auroc,
            'avg_precision': self.avg_precision
        }


def compute_metrics(labels: np.ndarray, preds: np.ndarray, probs: np.ndarray) -> Metrics:
    """
    Compute all classification metrics robustly.
    
    Args:
        labels: Ground truth (0=real, 1=fake)
        preds: Binary predictions
        probs: Probability scores for positive class (fake)
    
    Returns:
        Metrics dataclass with all computed values
    """
    metrics = Metrics(labels=labels, preds=preds, probs=probs)
    
    if len(labels) == 0:
        return metrics
    
    # Basic metrics with zero_division handling
    metrics.accuracy = accuracy_score(labels, preds)
    metrics.precision = precision_score(labels, preds, zero_division=0)
    metrics.recall = recall_score(labels, preds, zero_division=0)
    metrics.f1 = f1_score(labels, preds, zero_division=0)
    
    # Specificity = TN / (TN + FP)
    cm = confusion_matrix(labels, preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        metrics.specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    # AUROC and Average Precision (require both classes present)
    if len(np.unique(labels)) > 1:
        try:
            metrics.auroc = roc_auc_score(labels, probs)
            metrics.avg_precision = average_precision_score(labels, probs)
        except ValueError:
            metrics.auroc = float('nan')
            metrics.avg_precision = float('nan')
    else:
        metrics.auroc = float('nan')
        metrics.avg_precision = float('nan')
    
    return metrics

---
## 2. Luminance Dataset

In [ ]:
class LuminanceDataset(Dataset):
    """
    GPU-optimized dataset that only loads images and converts to luminance.
    Gradient computation is done on GPU inside the model.
    """
    def __init__(self, img_paths: list[str], labels: list[int], img_size=256):
        self.img_paths = img_paths
        self.labels = labels
        self.img_size = img_size
        self.resize = transforms.Resize((img_size, img_size))
        
        # BT.709 luminance coefficients
        self.R_COEFF = 0.2126
        self.G_COEFF = 0.7152
        self.B_COEFF = 0.0722

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        img = self.resize(img)
        img_tensor = transforms.functional.to_tensor(img)
        
        # RGB -> Luminance (BT.709)
        luminance = (self.R_COEFF * img_tensor[0] + 
                     self.G_COEFF * img_tensor[1] + 
                     self.B_COEFF * img_tensor[2])
        luminance = luminance.unsqueeze(0)  # (1, H, W)
        
        return luminance.float(), torch.tensor(self.labels[idx], dtype=torch.float32)

---
## 3. CNN Model with Discriminative Gradient Features

In [ ]:
class CompactGradientNet(nn.Module):
    def __init__(self, depth=4, base_filters=32, dropout=0.3, embedding_dim=128):
        """
        CNN for gradient field classification with discriminative features.
        5-channel input: [Gx, Gy, magnitude, angle, coherence]
        """
        super().__init__()
        
        # Sobel kernels
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], 
                               dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], 
                               dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)
        
        # Gaussian kernel for structure tensor smoothing
        gaussian = torch.tensor([[1, 4, 6, 4, 1], [4, 16, 24, 16, 4],
                                  [6, 24, 36, 24, 6], [4, 16, 24, 16, 4],
                                  [1, 4, 6, 4, 1]], dtype=torch.float32) / 256.0
        self.register_buffer('gaussian', gaussian.view(1, 1, 5, 5))

        # CNN layers
        layers = []
        in_ch = 5
        for i in range(depth):
            out_ch = base_filters * (2**i)
            layers.extend([
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ])
            if dropout > 0:
                layers.append(nn.Dropout2d(dropout))
            in_ch = out_ch

        self.cnn = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.embedding = nn.Linear(out_ch, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, 1)

    def compute_gradient_field(self, luminance):
        """Compute 5-channel gradient field on GPU."""
        G_x = F.conv2d(luminance, self.sobel_x, padding=1)
        G_y = F.conv2d(luminance, self.sobel_y, padding=1)
        
        magnitude = torch.sqrt(G_x**2 + G_y**2 + 1e-8)
        angle = torch.atan2(G_y, G_x) / math.pi
        
        # Structure tensor for coherence
        Gxx, Gxy, Gyy = G_x * G_x, G_x * G_y, G_y * G_y
        Sxx = F.conv2d(Gxx, self.gaussian, padding=2)
        Sxy = F.conv2d(Gxy, self.gaussian, padding=2)
        Syy = F.conv2d(Gyy, self.gaussian, padding=2)
        
        trace = Sxx + Syy
        det_term = torch.sqrt((Sxx - Syy)**2 + 4 * Sxy**2 + 1e-8)
        lambda1, lambda2 = 0.5 * (trace + det_term), 0.5 * (trace - det_term)
        coherence = ((lambda1 - lambda2) / (lambda1 + lambda2 + 1e-8))**2
        
        magnitude_scaled = torch.log1p(magnitude * 10)
        
        return torch.cat([G_x, G_y, magnitude_scaled, angle, coherence], dim=1)

    def forward(self, luminance):
        x = self.compute_gradient_field(luminance)
        x = self.cnn(x)
        x = self.global_pool(x).flatten(1)
        emb = self.embedding(x)
        logit = self.classifier(emb)
        return logit.squeeze(1), emb


---
## 4. Training and Evaluation Functions

In [ ]:
from tqdm import tqdm

def one_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    losses = []
    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)



In [ ]:
@torch.inference_mode()
def evaluate(model, loader, device, threshold=0.5) -> Metrics:
    """
    Evaluate model and return comprehensive Metrics object.
    
    Args:
        model: Trained model
        loader: DataLoader
        device: Device to use
        threshold: Classification threshold (default 0.5)
    
    Returns:
        Metrics object with all evaluation metrics
    """
    model.eval()
    all_labels, all_probs = [], []

    for x, y in tqdm(loader, desc="Evaluating", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits, _ = model(x)
        probs = torch.sigmoid(logits)
        all_labels.append(y.cpu())
        all_probs.append(probs.cpu())

    if len(all_labels) == 0:
        return Metrics()

    labels = torch.cat(all_labels).numpy()
    probs = torch.cat(all_probs).numpy().flatten()
    preds = (probs >= threshold).astype(int)
    
    return compute_metrics(labels, preds, probs)



---
## 5. Visualization Functions

In [ ]:
def plot_metrics_curves(metrics: Metrics, title_prefix=""):
    """
    Plot ROC curve, Precision-Recall curve, and probability distribution.
    
    Args:
        metrics: Metrics object with labels, preds, probs
        title_prefix: Optional prefix for plot titles
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    labels, probs = metrics.labels, metrics.probs
    
    # 1. ROC Curve
    if not np.isnan(metrics.auroc):
        fpr, tpr, thresholds = roc_curve(labels, probs)
        axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC = {metrics.auroc:.3f})')
        axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
        axes[0].fill_between(fpr, tpr, alpha=0.2)
        
        # Find optimal threshold (Youden's J)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        best_thresh = thresholds[best_idx]
        axes[0].scatter(fpr[best_idx], tpr[best_idx], s=100, c='red', 
                       label=f'Optimal (t={best_thresh:.2f})', zorder=5)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'{title_prefix}ROC Curve')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([0, 1])
    axes[0].set_ylim([0, 1])
    
    # 2. Precision-Recall Curve
    if not np.isnan(metrics.avg_precision):
        precision, recall, _ = precision_recall_curve(labels, probs)
        axes[1].plot(recall, precision, 'g-', lw=2, 
                    label=f'PR (AP = {metrics.avg_precision:.3f})')
        axes[1].fill_between(recall, precision, alpha=0.2, color='green')
        
        # Baseline (random classifier)
        baseline = labels.sum() / len(labels)
        axes[1].axhline(y=baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.2f})')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'{title_prefix}Precision-Recall Curve')
    axes[1].legend(loc='lower left')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([0, 1])
    axes[1].set_ylim([0, 1])
    
    # 3. Probability Distribution
    real_probs = probs[labels == 0]
    fake_probs = probs[labels == 1]
    
    axes[2].hist(real_probs, bins=50, alpha=0.6, color='green', label='Real', density=True)
    axes[2].hist(fake_probs, bins=50, alpha=0.6, color='red', label='Fake', density=True)
    axes[2].axvline(x=0.5, color='k', linestyle='--', lw=2, label='Threshold (0.5)')
    axes[2].set_xlabel('Predicted Probability (Fake)')
    axes[2].set_ylabel('Density')
    axes[2].set_title(f'{title_prefix}Probability Distribution')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_confusion_matrix(metrics: Metrics, title="Confusion Matrix"):
    """
    Plot confusion matrix with percentages.
    """
    cm = confusion_matrix(metrics.labels, metrics.preds)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Raw counts
    im = axes[0].imshow(cm, cmap='Blues')
    axes[0].set_title(f'{title} (Counts)')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(['Real', 'Fake'])
    axes[0].set_yticklabels(['Real', 'Fake'])
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=18,
                        color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.colorbar(im, ax=axes[0])
    
    # Percentages
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    im = axes[1].imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
    axes[1].set_title(f'{title} (Row-Normalized %)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(['Real', 'Fake'])
    axes[1].set_yticklabels(['Real', 'Fake'])
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f'{cm_pct[i, j]:.1f}%', ha='center', va='center', fontsize=18,
                        color='white' if cm_pct[i, j] > 50 else 'black')
    plt.colorbar(im, ax=axes[1])
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    tn, fp, fn, tp = cm.ravel()
    print(f"\n{'='*50}")
    print(f"CLASSIFICATION SUMMARY")
    print(f"{'='*50}")
    print(f"True Negatives (Real→Real):   {tn:5d}  ({tn/(tn+fp)*100:.1f}%)")
    print(f"False Positives (Real→Fake):  {fp:5d}  ({fp/(tn+fp)*100:.1f}%)")
    print(f"False Negatives (Fake→Real):  {fn:5d}  ({fn/(fn+tp)*100:.1f}%)")
    print(f"True Positives (Fake→Fake):   {tp:5d}  ({tp/(fn+tp)*100:.1f}%)")
    print(f"{'='*50}")

In [ ]:
def plot_training_history(history: dict):
    """
    Plot training curves from history dictionary.
    
    Args:
        history: Dict with 'train_loss' and 'val_metrics' (list of Metrics)
    """
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. Accuracy, F1
    accs = [m.accuracy for m in history['val_metrics']]
    f1s = [m.f1 for m in history['val_metrics']]
    axes[1].plot(epochs, accs, 'r-o', label='Accuracy', markersize=4)
    axes[1].plot(epochs, f1s, 'g-o', label='F1 Score', markersize=4)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Accuracy & F1')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1])
    
    # 3. AUROC, Average Precision
    aurocs = [m.auroc for m in history['val_metrics']]
    aps = [m.avg_precision for m in history['val_metrics']]
    axes[2].plot(epochs, aurocs, 'm-o', label='AUROC', markersize=4)
    axes[2].plot(epochs, aps, 'c-o', label='Avg Precision', markersize=4)
    
    best_auroc = max([x for x in aurocs if not np.isnan(x)], default=0)
    best_idx = aurocs.index(best_auroc) + 1
    axes[2].axvline(x=best_idx, color='k', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_idx})')
    
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].set_title(f'AUROC & AP (Best AUROC: {best_auroc:.3f})')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()

In [ ]:
def print_metrics_summary(metrics: Metrics, title="Metrics Summary"):
    """Print a nicely formatted metrics summary."""
    print(f"\n{'='*60}")
    print(f"{title:^60}")
    print(f"{'='*60}")
    print(f"{'Metric':<20} {'Value':>15} {'Description':>20}")
    print(f"{'-'*60}")
    print(f"{'Accuracy':<20} {metrics.accuracy:>15.4f} {'(TP+TN)/Total':>20}")
    print(f"{'Precision':<20} {metrics.precision:>15.4f} {'TP/(TP+FP)':>20}")
    print(f"{'Recall (Sensitivity)':<20} {metrics.recall:>15.4f} {'TP/(TP+FN)':>20}")
    print(f"{'Specificity':<20} {metrics.specificity:>15.4f} {'TN/(TN+FP)':>20}")
    print(f"{'F1 Score':<20} {metrics.f1:>15.4f} {'2*P*R/(P+R)':>20}")
    print(f"{'AUROC':<20} {metrics.auroc:>15.4f} {'Area under ROC':>20}")
    print(f"{'Avg Precision':<20} {metrics.avg_precision:>15.4f} {'Area under PR':>20}")
    print(f"{'='*60}\n")

---
## 6. Data Loading

In [ ]:
def get_image_paths_and_labels(folder):
    paths, labels = [], []
    for label, subfolder in enumerate(["real", "fake"]):
        subdir = os.path.join(folder, subfolder)
        for fname in os.listdir(subdir):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                paths.append(os.path.join(subdir, fname))
                labels.append(label)
    return paths, labels

# Load data
train_paths, train_labels = get_image_paths_and_labels(f"{data_dir}/train")
test_paths, test_labels = get_image_paths_and_labels(f"{data_dir}/test")

train_dataset = LuminanceDataset(train_paths, train_labels)
test_dataset = LuminanceDataset(test_paths, test_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                          num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} samples ({sum(train_labels)} fake, {len(train_labels)-sum(train_labels)} real)")
print(f"Test:  {len(test_dataset)} samples ({sum(test_labels)} fake, {len(test_labels)-sum(test_labels)} real)")
print(f"Device: {device}")

---
## 7. Training Loop

In [ ]:
def train_model():
    """Main training loop with comprehensive metrics tracking."""
    best_auroc = -1.0
    best_path = ARTIFACTS_DIR / "gradient_cnn_v2.pth"
    history = {'train_loss': [], 'val_metrics': []}
    patience_counter = 0

    # Load data
    train_paths, train_labels = get_image_paths_and_labels(f"{DATA_DIR}/train")
    test_paths, test_labels = get_image_paths_and_labels(f"{DATA_DIR}/test")

    train_dataset = LuminanceDataset(train_paths, train_labels)
    test_dataset = LuminanceDataset(test_paths, test_labels)

    # OPTIMIZED DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY,
        persistent_workers=True  # Avoid worker respawn overhead
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False,
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY,
        persistent_workers=True
    )

    print(f"Train: {len(train_dataset)} samples ({sum(train_labels)} fake, {len(train_labels)-sum(train_labels)} real)")
    print(f"Test:  {len(test_dataset)} samples ({sum(test_labels)} fake, {len(test_labels)-sum(test_labels)} real)")

    model = CompactGradientNet(depth=4, base_filters=32, dropout=0.3).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=True
    )

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Features: 5-channel [Gx, Gy, magnitude, angle, coherence]")
    print("=" * 80)

    for epoch in range(EPOCHS):
        t0 = time.time()
        tr_loss = one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_metrics = evaluate(model, test_loader, DEVICE)
        dt = time.time() - t0

        history['train_loss'].append(tr_loss)
        history['val_metrics'].append(val_metrics)
        scheduler.step(val_metrics.auroc)

        print(
            f"Epoch {epoch+1:2d}/{EPOCHS} | {dt:5.1f}s | "
            f"Loss: {tr_loss:.4f} | Acc: {val_metrics.accuracy:.4f} | "
            f"F1: {val_metrics.f1:.4f} | AUROC: {val_metrics.auroc:.4f} | "
            f"AP: {val_metrics.avg_precision:.4f}"
        )

        if not np.isnan(val_metrics.auroc) and val_metrics.auroc > best_auroc:
            best_auroc = val_metrics.auroc
            patience_counter = 0
            torch.save({"model_state": model.state_dict()}, best_path)
            print(f"  ✅ New best model saved (AUROC={best_auroc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  ⏹️ Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(torch.load(best_path)["model_state"])
    print(f"\n✅ Training complete. Best AUROC: {best_auroc:.4f}")
    
    return model, history


if __name__ == "__main__":
    # Mount drive in Colab
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("Not running in Colab, assuming local data paths")
    
    model, history = train_model()


---
## 8. Results Visualization

In [ ]:
# Training curves
plot_training_history(history)

In [ ]:
# Final evaluation on test set
final_metrics = evaluate(model, test_loader, device)

# Print summary
print_metrics_summary(final_metrics, "Final Test Set Metrics")

# Classification report
print("\nClassification Report:")
print(classification_report(final_metrics.labels, final_metrics.preds, 
                           target_names=['Real', 'Fake']))

In [ ]:
# Confusion matrix
plot_confusion_matrix(final_metrics, "Gradient Field CNN v2")

In [ ]:
# ROC, PR curves, and probability distribution
plot_metrics_curves(final_metrics, "Gradient Field CNN v2 - ")

---
## 9. Gradient Feature Visualization

In [ ]:
def visualize_gradient_features(dataset, model, device, index):
    """Visualize 5-channel gradient field for a sample."""
    model.eval()
    label = "REAL" if dataset.labels[index] == 0 else "FAKE"
    
    img = Image.open(dataset.img_paths[index]).convert('RGB')
    img = transforms.Resize((256, 256))(img)
    
    luminance, _ = dataset[index]
    with torch.no_grad():
        gf = model.compute_gradient_field(luminance.unsqueeze(0).to(device))[0].cpu().numpy()
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    axes[0,0].imshow(np.array(img))
    axes[0,0].set_title('Original')
    axes[0,0].axis('off')
    
    axes[0,1].imshow(luminance[0], cmap='gray')
    axes[0,1].set_title('Luminance')
    axes[0,1].axis('off')
    
    for i, (name, cmap) in enumerate([('Gx', 'RdBu'), ('Gy', 'RdBu'), 
                                       ('Magnitude', 'hot'), ('Angle', 'hsv'), ('Coherence', 'viridis')]):
        ax = axes[(i+2)//4, (i+2)%4]
        im = ax.imshow(gf[i], cmap=cmap)
        ax.set_title(name)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    axes[1,3].axis('off')
    plt.suptitle(f'{label} Sample - Gradient Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize real and fake samples
real_idx = next(i for i, l in enumerate(test_dataset.labels) if l == 0)
fake_idx = next(i for i, l in enumerate(test_dataset.labels) if l == 1)

visualize_gradient_features(test_dataset, model, device, real_idx)
visualize_gradient_features(test_dataset, model, device, fake_idx)